In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import os

In [ ]:
def save_all_roc_curves(dataset='Kd', base_path='/home/rlawlsgurjh/hdd/work/gitlab/LT-MMFE/cue/results'):
    labels = ['S(1uM)_label', 'S(100nM)_label']
    plot_output_dir = os.path.join(base_path, 'plots', dataset, 'ROC_Curves')
    os.makedirs(plot_output_dir, exist_ok=True)
    
    # 요청하신 이름 'CMFE'로 변경
    model_order = ['DeepDTA', 'GraphDTA', 'TEFDTA', 'HMM-DTA', 'ADMET-AI', 'MMFDL', 'CMFE']
    
    custom_palette = ["#8F6B9F", "#7275A7", "#5BA4BB", "#95CA73", "#DFDA70", "#EFBC65", "#FC979B"]
    
    model_configs = {
        'CMFE': {
            'type': 'cmfe',
            'path': '/home/rlawlsgurjh/hdd/work/gitlab/LT-MMFE/cue/results/{dataset}/{label}/v1/CMFE/fold{fold}/predictions.csv',
            'prob_col': 'mlp',
            'true_col': 'GroundTruth'
        },
        'ADMET-AI': {
            'type': 'admet',
            'path': '/home/rlawlsgurjh/hdd/work/gitlab/LT-MMFE/admet_ai/results/chemprop_rdkit/{dataset}/{label}/fold{fold}/test_preds.csv',
            'prob_col': 'y_score',
            'true_col': 'y_true'
        },
        'MMFDL': {
            'type': 'mmfdl',
            'path': '/home/rlawlsgurjh/hdd/work/gitlab/LT-MMFE/MMFDL/results/SGD/selectivity/{dataset}/{label}/v1/fold{fold}/predictions.csv',
            'prob_col': 'y_pred_prob',
            'true_col': 'y_true'
        },
        'DeepDTA':  {'type': 'dta', 'dta_name': 'deepdta'},
        'GraphDTA': {'type': 'dta', 'dta_name': 'graphdta'},
        'TEFDTA':   {'type': 'dta', 'dta_name': 'tefdta'},
        'HMM-DTA':  {'type': 'dta', 'dta_name': 'structurefree-dta'}
    }
    
    mean_fpr = np.linspace(0, 1, 100)
    
    for label in labels:
        plt.figure(figsize=(8, 6))
        
        for i, model_name in enumerate(model_order):
            config = model_configs[model_name]
            tprs = []
            aucs = []
            
            for fold in range(1, 6):
                try:
                    if config['type'] in ['cmfe', 'admet', 'mmfdl']:
                        file_path = config['path'].format(dataset=dataset, label=label, fold=fold)
                        df = pd.read_csv(file_path)
                        
                        valid_df = df[[config['true_col'], config['prob_col']]].dropna()
                        if len(valid_df) == 0: continue
                        
                        y_true = valid_df[config['true_col']]
                        y_prob = valid_df[config['prob_col']]
                    
                    elif config['type'] == 'dta':
                        gt_path = f'/home/rlawlsgurjh/hdd/work/gitlab/LT-MMFE/chembl/data/selectivity_df/{dataset}/fold{fold}/selectivity_test.csv'
                        dta_path = f'/home/rlawlsgurjh/hdd/work/gitlab/LT-MMFE/dta_results/results_original/{dataset}/{config["dta_name"]}/fold{fold}/pred_selectivity.csv'
                        
                        gt_df = pd.read_csv(gt_path)
                        dta_df = pd.read_csv(dta_path)
                        
                        prob_col = f"pred_{label}"
                        merged = pd.merge(gt_df[['compound_chembl_id', label]], 
                                          dta_df[['compound_chembl_id', prob_col]], 
                                          on='compound_chembl_id', how='inner')
                        
                        valid_df = merged.dropna()
                        if len(valid_df) == 0: continue
                        
                        y_true = valid_df[label]
                        y_prob = valid_df[prob_col]
                    
                    fpr, tpr, _ = roc_curve(y_true, y_prob)
                    roc_auc = auc(fpr, tpr)
                    aucs.append(roc_auc)
                    
                    interp_tpr = np.interp(mean_fpr, fpr, tpr)
                    interp_tpr[0] = 0.0
                    tprs.append(interp_tpr)
                    
                except Exception as e:
                    continue
            
            if len(tprs) > 0:
                mean_tpr = np.mean(tprs, axis=0)
                mean_tpr[-1] = 1.0
                
                mean_auc_val = np.mean(aucs)
                if config['type'] == 'cmfe':
                    std_auc_val = np.std(aucs, ddof=0) if len(aucs) > 1 else 0.0
                else:
                    std_auc_val = np.std(aucs, ddof=1) if len(aucs) > 1 else 0.0
                
                # CMFE로 조건 변경
                is_cmfe = (model_name == 'CMFE')
                linewidth = 3.0 if is_cmfe else 1.5
                alpha = 1.0 if is_cmfe else 0.8
                color = custom_palette[i]
                
                label_str = f'{model_name} (AUC = {mean_auc_val:.3f} $\pm$ {std_auc_val:.3f})'
                
                plt.plot(mean_fpr, mean_tpr, color=color,
                         label=label_str,
                         linewidth=linewidth, alpha=alpha)
        
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        # 1. 범례 없는 버전 저장
        save_path_no_leg = os.path.join(plot_output_dir, f"{dataset}_{label}_ROC.png")
        plt.savefig(save_path_no_leg, dpi=300)
        print(f"✅ 그래프 저장 완료 (범례 X): {save_path_no_leg}")
        
        # 2. 범례 있는 버전 저장
        plt.legend(loc="lower right", fontsize=14)
        save_path_with_leg = os.path.join(plot_output_dir, f"{dataset}_{label}_ROC_with_legend.png")
        plt.savefig(save_path_with_leg, dpi=300)
        print(f"✅ 그래프 저장 완료 (범례 O): {save_path_with_leg}")
        
        plt.close()

# 실행 
save_all_roc_curves(dataset='Kd')
save_all_roc_curves(dataset='Ki')

✅ 그래프 저장 완료 (범례 X): /home/rlawlsgurjh/hdd/work/gitlab/LT-MMFE/cmfe/results/plots/Kd/ROC_Curves/Kd_S(1uM)_label_ROC.png
✅ 그래프 저장 완료 (범례 O): /home/rlawlsgurjh/hdd/work/gitlab/LT-MMFE/cmfe/results/plots/Kd/ROC_Curves/Kd_S(1uM)_label_ROC_with_legend.png
✅ 그래프 저장 완료 (범례 X): /home/rlawlsgurjh/hdd/work/gitlab/LT-MMFE/cmfe/results/plots/Kd/ROC_Curves/Kd_S(100nM)_label_ROC.png
✅ 그래프 저장 완료 (범례 O): /home/rlawlsgurjh/hdd/work/gitlab/LT-MMFE/cmfe/results/plots/Kd/ROC_Curves/Kd_S(100nM)_label_ROC_with_legend.png
✅ 그래프 저장 완료 (범례 X): /home/rlawlsgurjh/hdd/work/gitlab/LT-MMFE/cmfe/results/plots/Ki/ROC_Curves/Ki_S(1uM)_label_ROC.png
✅ 그래프 저장 완료 (범례 O): /home/rlawlsgurjh/hdd/work/gitlab/LT-MMFE/cmfe/results/plots/Ki/ROC_Curves/Ki_S(1uM)_label_ROC_with_legend.png
✅ 그래프 저장 완료 (범례 X): /home/rlawlsgurjh/hdd/work/gitlab/LT-MMFE/cmfe/results/plots/Ki/ROC_Curves/Ki_S(100nM)_label_ROC.png
✅ 그래프 저장 완료 (범례 O): /home/rlawlsgurjh/hdd/work/gitlab/LT-MMFE/cmfe/results/plots/Ki/ROC_Curves/Ki_S(100nM)_label_ROC_with_leg